In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/01001
0


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  1562.7921867842688
RUN  2 , total integrated cost =  1450.2386314160265
RUN  3 , total integrated cost =  1341.2472828049858
RUN  4 , total integrated cost =  1235.5159974323078
RUN  5 , total integrated cost =  1135.277205904237
RUN  6 , total integrated cost =  1038.3740423155175
RUN  7 , total integrated cost =  948.4560142594171
RUN  8 , total integrated cost =  862.4601654726039
RUN  9 , total integrated cost =  780.136330455909
RUN  10 , total integrated cost =  705.4764713213925
RUN  11 , total integrated cost =  640.298952325234
RUN  12 , total integrated cost =  584.8369751521782
RUN  13 , total integrated cost =  535.1802680831643
RUN  14 , total integrated cost =  492.61411675068825
RUN  15 , total integrated cost =  454.4963257696394
RUN  16 

ERROR:root:Problem in initial value trasfer


RUN  142 , total integrated cost =  5028.362370025237
Improved over  142  iterations in  9.53118959441781  seconds by  1.3064188004045434  percent.
Problem in initial value trasfer:  Vmean_exc -56.624486214580344 -56.624483962664
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  3372.148774856119
RUN  2 , total integrated cost =  2440.556164811932
RUN  3 , total integrated cost =  1770.8660936432102
RUN  4 , total integrated cost =  1343.745295798499
RUN  5 , total integrated cost =  1034.1112976059073
RUN  6 , total integrated cost =  815.0820478268225
RUN  7 , total integrated cost =  643.5560674533381
RUN  8 , total integrated cost =  523.4090526665528
RUN  9 , total integrated cost =  441.77383017677715
RUN  10 , total integrated cost =  381.41815576110025
RUN  11 , total integrated cost =  338.926

ERROR:root:Problem in initial value trasfer


RUN  2000 , total integrated cost =  45.98687216655301
RUN  2000 , total integrated cost =  45.98687216655301
Improved over  2000  iterations in  139.26814089156687  seconds by  99.64674597867163  percent.
Problem in initial value trasfer:  Vmean_exc -56.67071325263559 -56.670710991329905
weight =  2830.8241084973615
set cost params:  1.0 2830.8241084973615 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13014.70492894458
Gradient descend method:  None
RUN  1 , total integrated cost =  12914.03145266772
RUN  2 , total integrated cost =  12913.205562935258
RUN  3 , total integrated cost =  12912.951232127485
RUN  4 , total integrated cost =  12906.084230916626
RUN  5 , total integrated cost =  12899.731151705128
RUN  6 , total integrated cost =  12899.6121848236
RUN  7 , total integrated cost =  12897.940897727522
RUN  8 , total integrated cost =  12896.812938651714
RUN  9 , total integrated cost =  12896.670672470953
RUN  10 , total integrated cost =  12895.

...

RUN  1 , total integrated cost =  23532.636131717038
RUN  2 , total integrated cost =  23532.636131717038
Control only changes marginally.
RUN  2 , total integrated cost =  23532.636131717038
Improved over  2  iterations in  0.22038866579532623  seconds by  4.834539879539079e-08  percent.
Problem in initial value trasfer:  Vmean_exc -73.654523026662 -73.65453894311506
weight =  10.00000000483454
set cost params:  1.0 10.00000000483454 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23532.636131717038
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  23532.636131717038
Control only changes marginally.
RUN  1 , total integrated cost =  23532.636131717038
Improved over  1  iterations in  0.1505990196019411  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -73.654523026662 -73.65453894311506
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  3886.419040859876
RUN  2 , total integrated cost =  340.2473621900726
RUN  3 , total integrated cost =  237.98127241629282
RUN  4 , total integrated cost =  206.10455903050058
RUN  5 , total integrated cost =  182.35888414444008
RUN  6 , total integrated cost =  136.0100998513903
RUN  7 , total integrated cost =  121.20805993554475
RUN  8 , total integrated cost =  115.99254489284364
RUN  9 , total integrated cost =  112.02102758770488
RUN  10 , total integrat

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  133 , total integrated cost =  32198.300106219205
Improved over  133  iterations in  9.140437450259924  seconds by  3.2639230771360133  percent.
Problem in initial value trasfer:  Vmean_exc -56.703542514990986 -56.70354249145949


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
------- 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  167 , total integrated cost =  149.01826728942976
Improved over  167  iterations in  11.593374159187078  seconds by  98.20535368435985  percent.
Problem in initial value trasfer:  Vmean_exc -56.639799857408974 -56.639799961949436
weight =  552.4092697628653
set cost params:  1.0 552.4092697628653 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8230.762608783454
Gradient descend method:  None
RUN  1 , total integrated cost =  8220.46996055297
RUN  2 , total integrated cost =  8220.32528354516
RUN  3 , total integrated cost =  8220.26124608399
RUN  4 , total integrated cost =  8220.250488232425
RUN  5 , total integrated cost =  8220.084660023786
RUN  6 , total integrated cost =  8219.98816533565
RUN  7 , total integrated cost =  8219.980602322237
RUN  8 , total integrated cost =  8215.896412757287
RUN  9 , total integrated cost =  8215.146509532571
RUN  10 , total integrated cost =  8215.145733825972
RUN  11 , total integr

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  26 , total integrated cost =  8215.14567139493
Improved over  26  iterations in  1.9750379659235477  seconds by  0.18973864428866705  percent.
Problem in initial value trasfer:  Vmean_exc -56.640288386102725 -56.6402766211353
-------  30 0.4250000000000001 0.5250000000000002
found solution for  30
-------  35 0.5500000000000003 0.5250000000000002
found solution for  35
-------  40 0.5250000000000001 0.5500000000000003
found solution for  40
-------  45 0.5000000000000002 0.5750000000000003
found solution for  45
-------  50 0.47500000000000014 0.6000000000000003
found solution for  50
-------  55 0.4250000000000001 0.6250000000000003
found solution for  55
-------  60 0.5500000000000003 0.6250000000000003
found solution for  60
-------  65 0.5000000000000002 0.6500000000000004
[0, 5, 10, 15, 20, 30, 35, 40, 45, 50, 55, 60] []
closest index  60
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total 

...

Control only changes marginally.
RUN  198 , total integrated cost =  180.99029686892732
Improved over  198  iterations in  13.036470236256719  seconds by  28.161808604791645  percent.
Problem in initial value trasfer:  Vmean_exc -56.679957916902254 -56.679957905241466
weight =  836.7164081327256
set cost params:  1.0 836.7164081327256 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  15142.573718681768
Gradient descend method:  None
RUN  1 , total integrated cost =  15132.494114605473
RUN  2 , total integrated cost =  15132.49186573375
RUN  3 , total integrated cost =  15132.491805739039
RUN  4 , total integrated cost =  15132.491799641906
RUN  5 , total integrated cost =  15132.491798364457
RUN  6 , total integrated cost =  15132.4917981046
RUN  7 , total integrated cost =  15132.49179798979
RUN  8 , total integrated cost =  15132.491797871902
RUN  9 , total integrated cost =  15132.491797845581
RUN  10 , total integrated cost =  15132.491797840317
RUN  11 , 

ERROR:root:Problem in initial value trasfer


RUN  14 , total integrated cost =  15132.491797838904
RUN  15 , total integrated cost =  15132.491797838902
RUN  16 , total integrated cost =  15132.491797838902
Control only changes marginally.
RUN  16 , total integrated cost =  15132.491797838902
Improved over  16  iterations in  1.174881599843502  seconds by  0.06657996870391969  percent.
Problem in initial value trasfer:  Vmean_exc -56.679907032067426 -56.67990830579474
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.7500000000000004
[0, 5, 10, 15, 20, 30, 35, 40, 45, 50, 55, 60, 70, 75, 80, 90, 100, 105, 110, 120, 130, 140, 145, 25] [80]
closest index  110
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  148.9689009835085
Gradient descend method:  None
RUN  1 , total integrated cost =  140.10285796589125
RUN  2 , total integrated cost =  135.2068111044417
RUN  3 , total integrated cost =  130.36068329983
RUN  4 , total in

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  25 , total integrated cost =  24055.935993535764
Improved over  25  iterations in  1.719946090131998  seconds by  0.2926092667053837  percent.
Problem in initial value trasfer:  Vmean_exc -56.70140709729082 -56.70140714182659
-------  100 0.4500000000000001 0.7750000000000005
-------  105 0.5750000000000002 0.7750000000000005
-------  110 0.5000000000000002 0.8000000000000005
-------  115 0.4250000000000001 0.8250000000000005
found solution for  115
-------  120 0.5500000000000003 0.8250000000000005
-------  125 0.47500000000000014 0.8500000000000005
found solution for  125
-------  130 0.6000000000000003 0.8500000000000005
-------  135 0.5250000000000001 0.8750000000000006
found solution for  135
-------  140 0.4500000000000001 0.9000000000000006
-------  145 0.5750000000000002 0.9000000000000006
------------------------------------------------------------
-------------------- 2
------------------------------------------------------------
found so

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  1031.4853362676404
set cost params:  1.0 1031.4853362676404 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5091.982199186547
Gradient descend method:  None
RUN  1 , total integrated cost =  5091.981305674036
RUN  2 , total integrated cost =  5091.981171534188
RUN  3 , total integrated cost =  5091.98113703312
RUN  4 , total integrated cost =  5091

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  26 , total integrated cost =  5091.981121693403
Improved over  26  iterations in  1.8008944112807512  seconds by  2.1160583486334872e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.62448564154964 -56.62448342327474
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  2880.112650970206
set cost params:  1.0 2880.112650970206 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13012.646497290814
Gradient descend method:  None
RUN  1 , total integrated cost =  13012.644938373205
RUN  2 , total integrated cost =  13012.6446359054
RUN  3 , total integrated cost =  13012.644523404142
RUN  4 , total integrated cost =  13012.644472841328
RUN  5 , total integrated cost =  13012.644449031439
RUN  6 , total integrated cost =  13012.644434267118
RUN  7 , total integrated cost =  13012.644425618133
RUN  8 , total integrated cost =  1

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  25 , total integrated cost =  13012.644385745794
Improved over  25  iterations in  1.6357783135026693  seconds by  1.622686838231857e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.67059444265533 -56.670595735675306
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  552.5363630618039
set cost params:  1.0 552.5363630618039 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8217.033496928088
Gradient descend method:  None
RUN  1 , total integrated cost =  8217.033496847836
RUN  2 , total integrated cost =  8217.03349682722
RUN  3 , total integrated cost =  8217.033496821667
RUN  4 , total integrated cost =  8217.033496820273
RUN  5 , total integrated cost =  8217.03349681995
RUN  6 , total integrated cost =  8217.033496819859
RUN  7 , total integrated cost =  8217.033496819857


...

RUN  0 , total integrated cost =  5092.353278190355
Gradient descend method:  None
RUN  1 , total integrated cost =  5092.353278190355
Control only changes marginally.
RUN  1 , total integrated cost =  5092.353278190355
Improved over  1  iterations in  0.18125995248556137  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.62448568958777 -56.62448347091536
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  30 0.4250000000000001 0.5250000000000002
converged for  30
-------  35 0.5500000000000003 0.5250000000000002
converged for  35
-------  40 0.5250000000000001 0.5500000000000003
converged for  40
-------  45 0.5000000000000002 0.5750000000000003
no convergence
weight =  3007.5598019850313
set cost params:  1.0 3007.5598019850313 0.0
i

ERROR:root:Problem in initial value trasfer


RUN  0 , total integrated cost =  20621.051487981378
Gradient descend method:  None
RUN  1 , total integrated cost =  20621.051487981378
Control only changes marginally.
RUN  1 , total integrated cost =  20621.051487981378
Improved over  1  iterations in  0.18463571928441525  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.69641834694706 -56.69641850569261
-------  50 0.47500000000000014 0.6000000000000003
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  60 0.5500000000000003 0.6250000000000003
converged for  60
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  70 0.4500000000000001 0.6750000000000004
converged for  70
-------  75 0.5750000000000002 0.6750000000000004
converged for  75
-------  80 0.5250000000000001 0.7000000000000004
converged for  80
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  90 0.6000000000000003 0.7250000000000004
converged for  90


ERROR:root:Problem in initial value trasfer


RUN  0 , total integrated cost =  23522.152876132946
Gradient descend method:  None
RUN  1 , total integrated cost =  23522.152876132946
Control only changes marginally.
RUN  1 , total integrated cost =  23522.152876132946
Improved over  1  iterations in  0.18647175282239914  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70067428485519 -56.70067434498127
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
[[True, True], [False, False], [True, True], [True, False], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True]]
-------  0 0.4000000000000001 0.35000000000000

In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [ ]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  60.925321626225696
Gradient descend method:  None
RUN  1 , total integrated cost =  38.423635440316694
RUN  2 , total integrated cost =  38.08480815823706
RUN  3 , total integrated cost =  37.747325602491046
RUN  4 , total integrated cost =  37.528812794880174
RUN  5 , total integrated cost =  37.291221514400455
RUN  6 , total integrated cost =  37.10895303526638
RUN  7 , total integrated cost =  36.89574339723048
RUN  8 , total integrated cost =  36.72938931034952
RUN  9 , total integrated cost =  36.528627514345736
RUN  10 , total integrated cost =  36.366055285893154
RUN  11 , total integrated cost =  36.146487095491814
RUN  12 , total integrated cost =  35.948447369520395
RUN  13 , total integrated cost =  35.65665398121099
RUN  14 , total integrated cost =  35.393659643091354
RUN  15 , total integrated cost =  34.81479892624267
RU

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  480 , total integrated cost =  356.3962911331163
Improved over  480  iterations in  92.23183918371797  seconds by  26.49808766956015  percent.
Problem in initial value trasfer:  Vmean_exc -56.62444043941683 -56.6244416510316
weight =  1429.2308848370851
set cost params:  1.0 1429.2308848370851 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5092.296540628952
Gradient descend method:  None
RUN  1 , total integrated cost =  5075.165466153191
RUN  2 , total integrated cost =  5065.727826614528
RUN  3 , total integrated cost =  5051.183372867451
RUN  4 , total integrated cost =  5041.844567308956
RUN  5 , total integrated cost =  5028.958219582492
RUN  6 , total integrated cost =  5015.320040159307
RUN  7 , total integrated cost =  4994.569252719049
RUN  8 , total integrated cost =  4985.445206960434
RUN  9 , total integrated cost =  4972.620159198732
RUN  10 , total integrated cost =  4962.497715001556
RUN  11 , total integ

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  851 , total integrated cost =  2543.1650318196675
Improved over  851  iterations in  101.8328967988491  seconds by  50.05858336157383  percent.
Problem in initial value trasfer:  Vmean_exc -56.62447256639118 -56.62447137372782
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  83.46730298230786
Gradient descend method:  None
RUN  1 , total integrated cost =  31.400401409016087
RUN  2 , total integrated cost =  31.290164363273114
RUN  3 , total integrated cost =  31.198730940876736
RUN  4 , total integrated cost =  31.119691795476413
RUN  5 , total integrated cost =  31.04770558552097
RUN  6 , total integrated cost =  30.988106751103455
RUN  7 , total integrated cost =  30.933049441665265
RUN  8 , total integrated cost =  30.883242697220847
RUN  9 , total integrated cost =  30.83728256291873
RUN  10 , total integrated cost =  30.79658347789384
RUN  

...

RUN  1000 , total integrated cost =  30.123037622836396
RUN  1000 , total integrated cost =  30.123037622836396
Improved over  1000  iterations in  221.52481520548463  seconds by  63.91037382719623  percent.
Problem in initial value trasfer:  Vmean_exc -56.670673866985005 -56.670673883922994
weight =  100
set cost params:  1.0 100.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  453.6635333053
Gradient descend method:  HS
RUN  1 , total integrated cost =  453.4249918424524
RUN  2 , total integrated cost =  453.23492375946995
RUN  3 , total integrated cost =  453.0055092192402
RUN  4 , total integrated cost =  452.78341185770296
RUN  5 , total integrated cost =  452.6732304651657
RUN  6 , total integrated cost =  452.4262728291841
RUN  7 , total integrated cost =  452.4191662036233
RUN  8 , total integrated cost =  452.1146310001635
RUN  9 , total integrated cost =  451.84320680460047
RUN  10 , total integrated cost =  451.7521323323293
RUN  11 , total integ

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  88 , total integrated cost =  437.848667434902
Improved over  88  iterations in  17.404879093170166  seconds by  3.4860341881954042  percent.
Problem in initial value trasfer:  Vmean_exc -56.670597921206614 -56.670601480044404
weight =  2972.1904213873036
set cost params:  1.0 2972.1904213873036 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13010.048586261906
Gradient descend method:  None
RUN  1 , total integrated cost =  12939.909231701216
RUN  2 , total integrated cost =  12909.937232237677
RUN  3 , total integrated cost =  12843.819529733148
RUN  4 , total integrated cost =  12813.104386091163
RUN  5 , total integrated cost =  12755.337343343717
RUN  6 , total integrated cost =  12726.262494099494
RUN  7 , total integrated cost =  12672.877421486999
RUN  8 , total integrated cost =  12641.727430954123
RUN  9 , total integrated cost =  12586.83749835107
RUN  10 , total integrated cost =  12558.289013849379
RUN  11 ,

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  756 , total integrated cost =  6213.819501595311
Improved over  756  iterations in  88.52555642835796  seconds by  52.23830671810975  percent.
Problem in initial value trasfer:  Vmean_exc -56.6706365821463 -56.670637168819056
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  64.24562991274853
Gradient descend method:  None
RUN  1 , total integrated cost =  56.65853804556468
RUN  2 , total integrated cost =  56.61670712736037
RUN  3 , total integrated cost =  56.583440053690424
RUN  4 , total integrated cost =  56.52734722521562
RUN  5 , total integrated cost =  56.48087567914377
RUN  6 , total integrated cost =  56.409516319249754
RUN  7 , total integrated cost =  56.348131684471156
RUN  8 , total integrated cost =  56.235006941547965
RUN  9 , total integrated cost =  56.12599523908042
RUN  10 , total integrated cost =  54.84035958616884
RUN  11 ,

In [ ]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

In [ ]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

In [ ]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

In [ ]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [ ]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

In [ ]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

In [ ]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

In [ ]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

In [ ]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)